# Community Proposal Board — Bài thực hành Blockchain
**Sinh viên:** Võ Khắc Tuyên · **MSSV:** 30219042961  
**Môn:** DS 441 A — Blockchain: Theory & Practices

Notebook kết nối smart contract Solidity trên Ganache bằng web3.py. Output đã lưu thể hiện giao dịch thật, receipt, event, lỗi bị từ chối và trạng thái cuối.

Chạy lại: bật Ganache, chạy `npm run deploy`, cài `requirements-notebook.txt`, chọn kernel Python (Community Board), rồi **Restart Kernel and Run All Cells**. Mỗi lần chạy tạo đề xuất mới. Chỉ dùng Ether và tài khoản thử nghiệm. ABI và cấu hình được nhúng trong cell; nếu có `deployment.json`, notebook ưu tiên cấu hình mới nhất. GitHub có thể hiển thị output đã lưu mà không chạy Ganache.

## 1. Kết nối và kiểm tra cấu hình
Xác thực mạng Ganache, Chain ID, địa chỉ có bytecode và ABI. Dùng ba địa chỉ A/B/C làm tác giả, người ủng hộ và người chưa ủng hộ.

In [1]:
import json
import sys
from pathlib import Path
from urllib.parse import urlparse
from importlib.metadata import version
from web3 import Web3
from web3.exceptions import Web3Exception

EMBEDDED_CONFIG = json.loads(r'''{
  "contractName": "CommunityProposalBoard",
  "address": "0x1661710adE37BEEBb806aAA6643E4cd281d6F165",
  "chainId": 1337,
  "rpcUrl": "http://127.0.0.1:7545",
  "abi": [
    {
      "anonymous": false,
      "inputs": [
        {
          "indexed": true,
          "internalType": "uint256",
          "name": "proposalId",
          "type": "uint256"
        },
        {
          "indexed": true,
          "internalType": "address",
          "name": "author",
          "type": "address"
        }
      ],
      "name": "ProposalClosed",
      "type": "event"
    },
    {
      "anonymous": false,
      "inputs": [
        {
          "indexed": true,
          "internalType": "uint256",
          "name": "proposalId",
          "type": "uint256"
        },
        {
          "indexed": true,
          "internalType": "address",
          "name": "author",
          "type": "address"
        },
        {
          "indexed": false,
          "internalType": "string",
          "name": "title",
          "type": "string"
        }
      ],
      "name": "ProposalCreated",
      "type": "event"
    },
    {
      "anonymous": false,
      "inputs": [
        {
          "indexed": true,
          "internalType": "uint256",
          "name": "proposalId",
          "type": "uint256"
        },
        {
          "indexed": true,
          "internalType": "address",
          "name": "supporter",
          "type": "address"
        },
        {
          "indexed": false,
          "internalType": "uint256",
          "name": "endorsementCount",
          "type": "uint256"
        }
      ],
      "name": "ProposalEndorsed",
      "type": "event"
    },
    {
      "inputs": [
        {
          "internalType": "uint256",
          "name": "proposalId",
          "type": "uint256"
        }
      ],
      "name": "closeProposal",
      "outputs": [],
      "stateMutability": "nonpayable",
      "type": "function"
    },
    {
      "inputs": [
        {
          "internalType": "string",
          "name": "title",
          "type": "string"
        },
        {
          "internalType": "string",
          "name": "description",
          "type": "string"
        }
      ],
      "name": "createProposal",
      "outputs": [
        {
          "internalType": "uint256",
          "name": "id",
          "type": "uint256"
        }
      ],
      "stateMutability": "nonpayable",
      "type": "function"
    },
    {
      "inputs": [
        {
          "internalType": "uint256",
          "name": "proposalId",
          "type": "uint256"
        }
      ],
      "name": "endorseProposal",
      "outputs": [],
      "stateMutability": "nonpayable",
      "type": "function"
    },
    {
      "inputs": [
        {
          "internalType": "uint256",
          "name": "proposalId",
          "type": "uint256"
        }
      ],
      "name": "getProposal",
      "outputs": [
        {
          "components": [
            {
              "internalType": "uint256",
              "name": "id",
              "type": "uint256"
            },
            {
              "internalType": "address",
              "name": "author",
              "type": "address"
            },
            {
              "internalType": "string",
              "name": "title",
              "type": "string"
            },
            {
              "internalType": "string",
              "name": "description",
              "type": "string"
            },
            {
              "internalType": "uint256",
              "name": "endorsementCount",
              "type": "uint256"
            },
            {
              "internalType": "bool",
              "name": "closed",
              "type": "bool"
            }
          ],
          "internalType": "struct CommunityProposalBoard.Proposal",
          "name": "",
          "type": "tuple"
        }
      ],
      "stateMutability": "view",
      "type": "function"
    },
    {
      "inputs": [
        {
          "internalType": "uint256",
          "name": "",
          "type": "uint256"
        },
        {
          "internalType": "address",
          "name": "",
          "type": "address"
        }
      ],
      "name": "hasEndorsed",
      "outputs": [
        {
          "internalType": "bool",
          "name": "",
          "type": "bool"
        }
      ],
      "stateMutability": "view",
      "type": "function"
    },
    {
      "inputs": [],
      "name": "proposalCount",
      "outputs": [
        {
          "internalType": "uint256",
          "name": "",
          "type": "uint256"
        }
      ],
      "stateMutability": "view",
      "type": "function"
    }
  ],
  "transactionHash": "0xb00c6332e8b7578d2ef221805351c7d6a2342b83051ff2a1453de39cc769fb6a"
}''')
config_path = Path("deployment.json")
config = json.loads(config_path.read_text(encoding="utf-8")) if config_path.exists() else EMBEDDED_CONFIG
RPC_URL = config["rpcUrl"]
CONTRACT_ADDRESS = config["address"]
ABI = config["abi"]

assert urlparse(RPC_URL).hostname in {"localhost", "127.0.0.1", "::1"}, "Use local Ganache only"
w3 = Web3(Web3.HTTPProvider(RPC_URL, request_kwargs={"timeout": 10}))
assert w3.is_connected(), f"Cannot connect to Ganache: {RPC_URL}"
assert any(name in w3.client_version.lower() for name in ("ganache", "testrpc")), "Expected Ganache"
assert Web3.is_address(CONTRACT_ADDRESS), "Invalid contract address"
assert isinstance(ABI, list), "ABI must be a JSON array"
address = Web3.to_checksum_address(CONTRACT_ADDRESS)
assert w3.eth.chain_id == int(config["chainId"]), "Wrong Chain ID"
assert w3.eth.get_code(address), "Contract missing. Run npm run deploy again."
contract = w3.eth.contract(address=address, abi=ABI)
accounts = w3.eth.accounts
assert len(accounts) >= 3, "Need at least three unlocked Ganache accounts"
A, B, C = accounts[:3]
results = []
print("Connected to Ganache and CommunityProposalBoard!")
print("Python:", sys.version.split()[0], "| web3.py:", version("web3"))
print("Client:", w3.client_version)
print("RPC:", RPC_URL)
print("Chain ID:", w3.eth.chain_id)
print("Contract:", address)
print("ABI entries:", len(ABI))
print("Proposal count before demo:", contract.functions.proposalCount().call())
for role, account in zip(("A - author", "B - supporter", "C - newcomer"), (A, B, C)):
    print(role, ":", account, "| test ETH:", w3.from_wei(w3.eth.get_balance(account), "ether"))

Connected to Ganache and CommunityProposalBoard!
Python: 3.11.7 | web3.py: 7.16.0
Client: Ganache/v7.7.3/EthereumJS TestRPC/v7.7.3/ethereum-js
RPC: http://127.0.0.1:7545
Chain ID: 1337
Contract: 0x1661710adE37BEEBb806aAA6643E4cd281d6F165
ABI entries: 9
Proposal count before demo: 3
A - author : 0x195Cab71A742e0Bb60a756cDa133767Cf95E1073 | test ETH: 99.998183785736962817
B - supporter : 0xA2C5a72040C6951aD1B83224685457802f33b528 | test ETH: 99.999900219080795522
C - newcomer : 0x7C1cfe07cbdDc532061840591F58642EF6ebfE9D | test ETH: 99.999992654089983553


## 2. Hàm hỗ trợ
`send` gửi giao dịch thật và kiểm tra receipt thành công. `reject` kiểm tra lý do revert bằng eth_call, sau đó gửi giao dịch thật với gas cố định và yêu cầu receipt **status = 0**. Giao dịch thất bại vẫn được ghi trong block và tốn gas thử nghiệm, nhưng thay đổi dữ liệu và event bị hoàn tác.

In [2]:
def passed(label):
    results.append((label, "PASS"))
    print("PASS:", label)

def send(label, function, sender, event_name=None):
    tx_hash = function.transact({"from": sender})
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=120)
    print("Transaction:", Web3.to_hex(tx_hash))
    print("Block:", receipt.blockNumber, "| status:", receipt.status, "| gas:", receipt.gasUsed)
    assert receipt.status == 1, f"Transaction failed: {label}"
    if event_name:
        logs = getattr(contract.events, event_name)().process_receipt(receipt)
        assert len(logs) == 1, f"Expected event {event_name}"
        print("Event:", event_name, dict(logs[0]["args"]))
    passed(label)
    return receipt

def proposal_data(proposal_id):
    fields = ("id", "author", "title", "description", "endorsementCount", "closed")
    return dict(zip(fields, contract.functions.getProposal(proposal_id).call()))

def show(proposal_id):
    print(json.dumps(proposal_data(proposal_id), ensure_ascii=False, indent=2))

def reject(label, function, sender, expected_reason):
    try:
        function.call({"from": sender})
    except (Web3Exception, ValueError) as exc:
        message = str(exc)
        assert expected_reason in message and "revert" in message.lower(), message
        print("Expected reason:", expected_reason)
        rpc_error = (getattr(exc, "rpc_response", None) or {}).get("error", {})
        print("Contract error:", rpc_error.get("message", message))
    else:
        raise AssertionError("Invalid action was accepted: " + label)
    try:
        tx_hash = function.transact({"from": sender, "gas": 250000})
    except Web3Exception as exc:
        # Ganache GUI may return a JSON-RPC error even after mining the revert.
        error = (getattr(exc, "rpc_response", None) or {}).get("error", {})
        data = error.get("data", {})
        if not isinstance(data, dict) or not data.get("hash"):
            raise
        assert expected_reason in str(exc), str(exc)
        tx_hash = Web3.to_bytes(hexstr=data["hash"])
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=120)
    print("Rejected transaction:", Web3.to_hex(tx_hash))
    print("Block:", receipt.blockNumber, "| status:", receipt.status, "| gas:", receipt.gasUsed)
    assert receipt.status == 0, "Invalid transaction must revert"
    assert len(receipt.logs) == 0, "Reverted transaction must not retain events"
    passed(label)

## 3. A tạo đề xuất
Lấy ID từ event ProposalCreated để notebook có thể chạy lại trên blockchain đã có dữ liệu. Kiểm tra tác giả, số lượt ban đầu bằng 0 và trạng thái đang mở.

In [3]:
receipt = send(
    "A creates a proposal",
    contract.functions.createProposal(
        "Trồng cây tại công viên",
        "Trồng thêm cây để tạo bóng mát và cải thiện không gian xanh cho cộng đồng."
    ),
    A,
    "ProposalCreated"
)
proposal_id = contract.events.ProposalCreated().process_receipt(receipt)[0]["args"]["proposalId"]
initial = proposal_data(proposal_id)
assert initial["author"] == A and initial["endorsementCount"] == 0 and initial["closed"] is False
print("New proposal ID:", proposal_id)
show(proposal_id)

Transaction: 0xb42ac341c2ae03617ace79026ea97c6f4fc8aa9ad36f75e4bcfced8c250f3a8a
Block: 20 | status: 1 | gas: 193014
Event: ProposalCreated {'proposalId': 3, 'author': '0x195Cab71A742e0Bb60a756cDa133767Cf95E1073', 'title': 'Trồng cây tại công viên'}
PASS: A creates a proposal
New proposal ID: 3
{
  "id": 3,
  "author": "0x195Cab71A742e0Bb60a756cDa133767Cf95E1073",
  "title": "Trồng cây tại công viên",
  "description": "Trồng thêm cây để tạo bóng mát và cải thiện không gian xanh cho cộng đồng.",
  "endorsementCount": 0,
  "closed": false
}


## 4. Chặn tác giả tự ủng hộ
A tự ủng hộ phải bị contract từ chối. Số lượt vẫn bằng 0 và mapping không ghi nhận lượt ủng hộ.

In [4]:
reject("Self endorsement rejected", contract.functions.endorseProposal(proposal_id), A, "Author cannot endorse own proposal")
assert proposal_data(proposal_id)["endorsementCount"] == 0
assert not contract.functions.hasEndorsed(proposal_id, A).call()

Expected reason: Author cannot endorse own proposal
Contract error: VM Exception while processing transaction: revert Author cannot endorse own proposal
Rejected transaction: 0x7a75b250e61eeeff792c96344cad14adb5d5c64f950029ff9aaa75c5bb2d196a
Block: 21 | status: 0 | gas: 28073
PASS: Self endorsement rejected


## 5. B ủng hộ thành công
Số lượt tăng lên 1; mapping ghi nhận B đã ủng hộ và event ProposalEndorsed được phát.

In [5]:
send("B endorses the proposal", contract.functions.endorseProposal(proposal_id), B, "ProposalEndorsed")
assert proposal_data(proposal_id)["endorsementCount"] == 1
assert contract.functions.hasEndorsed(proposal_id, B).call()
show(proposal_id)

Transaction: 0x9e2feeac6f8aae8072505beea660d2997675865e06904cb9a3e419952e746906
Block: 22 | status: 1 | gas: 74612
Event: ProposalEndorsed {'proposalId': 3, 'supporter': '0xA2C5a72040C6951aD1B83224685457802f33b528', 'endorsementCount': 1}
PASS: B endorses the proposal
{
  "id": 3,
  "author": "0x195Cab71A742e0Bb60a756cDa133767Cf95E1073",
  "title": "Trồng cây tại công viên",
  "description": "Trồng thêm cây để tạo bóng mát và cải thiện không gian xanh cho cộng đồng.",
  "endorsementCount": 1,
  "closed": false
}


## 6. Chặn ủng hộ trùng
B ủng hộ lần nữa phải bị từ chối với lý do Already endorsed. Số lượt giữ nguyên bằng 1.

In [6]:
reject("Duplicate endorsement rejected", contract.functions.endorseProposal(proposal_id), B, "Already endorsed")
assert proposal_data(proposal_id)["endorsementCount"] == 1

Expected reason: Already endorsed
Contract error: VM Exception while processing transaction: revert Already endorsed
Rejected transaction: 0x648fc6d448494a0752b6c8d299ac3345860b4f9b5b358b3d069d6ca61adffca4
Block: 23 | status: 0 | gas: 30318
PASS: Duplicate endorsement rejected


## 7. Chặn người khác đóng đề xuất
B không phải tác giả nên contract từ chối. Đề xuất vẫn đang mở.

In [7]:
reject("Unauthorized close rejected", contract.functions.closeProposal(proposal_id), B, "Only author can close proposal")
assert proposal_data(proposal_id)["closed"] is False

Expected reason: Only author can close proposal
Contract error: VM Exception while processing transaction: revert Only author can close proposal
Rejected transaction: 0xfaa53faa170e70c620801922547e72ae7a54fbe6b6492744e3f27a9a9667307b
Block: 24 | status: 0 | gas: 25860
PASS: Unauthorized close rejected


## 8. A đóng đề xuất
Giao dịch thành công, phát event ProposalClosed và trạng thái closed chuyển thành true.

In [8]:
send("A closes the proposal", contract.functions.closeProposal(proposal_id), A, "ProposalClosed")
assert proposal_data(proposal_id)["closed"] is True
show(proposal_id)

Transaction: 0x6ecc2fd310daec9eb33c762a5f8be44b28c9362effb7492483923ecccdc7c520
Block: 25 | status: 1 | gas: 49550


Event: ProposalClosed {'proposalId': 3, 'author': '0x195Cab71A742e0Bb60a756cDa133767Cf95E1073'}
PASS: A closes the proposal
{
  "id": 3,
  "author": "0x195Cab71A742e0Bb60a756cDa133767Cf95E1073",
  "title": "Trồng cây tại công viên",
  "description": "Trồng thêm cây để tạo bóng mát và cải thiện không gian xanh cho cộng đồng.",
  "endorsementCount": 1,
  "closed": true
}


## 9. Chặn ủng hộ sau khi đóng
Dùng C chưa từng ủng hộ để chứng minh lỗi do đề xuất đã đóng. Đồng thời kiểm tra tác giả không thể đóng lần thứ hai.

In [9]:
assert not contract.functions.hasEndorsed(proposal_id, C).call()
reject("Endorsement after closure rejected", contract.functions.endorseProposal(proposal_id), C, "Proposal is closed")
assert not contract.functions.hasEndorsed(proposal_id, C).call()
reject("Repeated closure rejected", contract.functions.closeProposal(proposal_id), A, "Proposal is closed")

Expected reason: Proposal is closed
Contract error: VM Exception while processing transaction: revert Proposal is closed
Rejected transaction: 0xf16950be667437b3f761eb08a7ea5609b72799a653e072fa51199895371c9ac0
Block: 26 | status: 0 | gas: 25909
PASS: Endorsement after closure rejected
Expected reason: Proposal is closed
Contract error: VM Exception while processing transaction: revert Proposal is closed


Rejected transaction: 0x856f64a77335427e9f6bd14fe7d1f89d1dbc9815eae7b53884ccf3d1a00ebd71
Block: 27 | status: 0 | gas: 27998
PASS: Repeated closure rejected


## 10. Kiểm tra đầu vào không hợp lệ
Contract phải chặn ID không tồn tại và tiêu đề rỗng. Thao tác tạo đề xuất không hợp lệ không được làm tăng tổng số đề xuất.

In [10]:
invalid_id = contract.functions.proposalCount().call() + 1000
reject("Unknown proposal rejected", contract.functions.endorseProposal(invalid_id), C, "Proposal does not exist")
count_before = contract.functions.proposalCount().call()
reject("Empty title rejected", contract.functions.createProposal("", "Description"), A, "Title must be 1-120 bytes")
assert contract.functions.proposalCount().call() == count_before

Expected reason: Proposal does not exist
Contract error: VM Exception while processing transaction: revert Proposal does not exist
Rejected transaction: 0xc1e26de25c3d881e8cac35f0758fa711d241cd6bc39f672ca130090dbf8eafe7
Block: 28 | status: 0 | gas: 23735
PASS: Unknown proposal rejected


Expected reason: Title must be 1-120 bytes
Contract error: VM Exception while processing transaction: revert Title must be 1-120 bytes
Rejected transaction: 0x6ffba3335b2ba15855f1687645d31c2389cc1f602822b1a789882fb1731137ef
Block: 29 | status: 0 | gas: 22700
PASS: Empty title rejected


## 11. Bảng kết quả và trạng thái cuối
Đề xuất thuộc A, có đúng 1 lượt từ B, đã đóng; C chưa ủng hộ. Mỗi dòng PASS chỉ được thêm sau khi kiểm tra giao dịch tương ứng thành công. Bất kỳ assertion thất bại nào đều dừng notebook.

In [11]:
final = proposal_data(proposal_id)
assert final["author"] == A
assert final["endorsementCount"] == 1
assert final["closed"] is True
assert contract.functions.hasEndorsed(proposal_id, B).call()
assert not contract.functions.hasEndorsed(proposal_id, C).call()
passed("Final state and endorsement mappings are correct")
show(proposal_id)
print("=" * 72)
for index, (label, outcome) in enumerate(results, 1):
    print(f"{index:02}. {outcome:4} | {label}")
print("=" * 72)
assert len(results) == 11
print(f"RESULT: {len(results)}/{len(results)} checks passed.")
print("All transaction hashes, receipts and events above were obtained from Ganache.")

PASS: Final state and endorsement mappings are correct


{
  "id": 3,
  "author": "0x195Cab71A742e0Bb60a756cDa133767Cf95E1073",
  "title": "Trồng cây tại công viên",
  "description": "Trồng thêm cây để tạo bóng mát và cải thiện không gian xanh cho cộng đồng.",
  "endorsementCount": 1,
  "closed": true
}
01. PASS | A creates a proposal
02. PASS | Self endorsement rejected
03. PASS | B endorses the proposal
04. PASS | Duplicate endorsement rejected
05. PASS | Unauthorized close rejected
06. PASS | A closes the proposal
07. PASS | Endorsement after closure rejected
08. PASS | Repeated closure rejected
09. PASS | Unknown proposal rejected
10. PASS | Empty title rejected
11. PASS | Final state and endorsement mappings are correct
RESULT: 11/11 checks passed.
All transaction hashes, receipts and events above were obtained from Ganache.


## Giải thích và nguồn tham khảo
- Quyền được kiểm tra trong Solidity bằng msg.sender, không phụ thuộc giao diện.
- Mapping hasEndorsed giới hạn một địa chỉ một lượt trên từng đề xuất.
- Chỉ tác giả được đóng; khi đóng, contract ngừng nhận ủng hộ.
- Dữ liệu output lấy từ Ganache cục bộ. Không chứa private key hoặc tiền thật.
- [web3.py](https://web3py.readthedocs.io/en/stable/overview.html), [Solidity](https://docs.soliditylang.org/), [Ganache](https://github.com/trufflesuite/ganache).
- Mã nguồn, notebook và tài liệu được xây dựng với sự hỗ trợ của OpenAI Codex. Sinh viên cần hiểu mã và tự giải thích trước khi nộp.